# MODELO SARIMAX PARA EL CONTAMINANTE CO PARA MADRID

En este *notebook*, ajustaremos modelos **SARIMAX (Seasonal AutoRegressive Integrated Moving Average with eXogenous variables)** para la predicción de las concentraciones del CO en Madrid, incorporando variables exógenas que pueden aportar información adicional sobre su evolución.

Los modelos SARIMAX constituyen una extensión de los modelos SARIMA, incorporando variables explicativas externas a la propia serie temporal. Se dice que ${X_t}$ sigue un modelo SARIMAX de órdenes $(p,d,q)\times(P,D,Q)_s$ si puede expresarse como:

$$
\Phi_P(B^s)\phi_p(B)\nabla_s^D\nabla^d
\left(
X_t-\sum_{i=1}^{k}\beta_i Z_{i,t}
\right)
=

\theta_q(B)\Theta_Q(B^s)Y_t,
$$

donde:

* ${Y_t} \sim WN(0,\sigma^2)$ es un proceso de ruido blanco.
* $\nabla_s^D=(1-B^s)^D$ representa la diferenciación estacional.
* $\nabla^d=(1-B)^d$ representa la diferenciación regular.
* $\Phi_P(B^s)=1-\Phi_1B^s-\dots-\Phi_PB^{sP}$ es el operador autorregresivo estacional (AR).
* $\phi_p(B)=1-\phi_1B-\dots-\phi_pB^p$ es el operador autorregresivo no estacional (AR).
* $\Theta_Q(B^s)=1-\Theta_1B^s-\dots-\Theta_QB^{sQ}$ es el operador de media móvil estacional (MA).
* $\theta_q(B)=1-\theta_1B-\dots-\theta_qB^q$ es el operador de media móvil no estacional (MA).
* $Z_{i,t}$ representa la $i$-ésima variable exógena en el instante $t$.
* $\beta_i$ representa el coeficiente asociado a la variable exógena $Z_{i,t}$.
* $s$ representa la periodicidad de la componente estacional.

De esta forma, el modelo SARIMAX combina la información contenida en la propia dinámica temporal de la serie con la proporcionada por las variables exógenas. Para realizar predicciones, será necesario disponer de los valores conocidos o estimados de estas variables para el horizonte temporal que se desea predecir.

Importamos las librerías y definimos las rutas.

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
warnings.filterwarnings("ignore")

from pmdarima import auto_arima
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn import metrics

from pylab import rcParams
plt.style.use("fivethirtyeight")
plt.rcParams["lines.linewidth"] = 1.5
light_style = {
    "figure.facecolor": "#d9effb",   
    "axes.facecolor": "#d9effb",
    "savefig.facecolor": "#d9effb",
    "axes.grid": True,
    "axes.grid.which": "both",
    "axes.spines.left": True,
    "axes.spines.right": True,
    "axes.spines.top": True,
    "axes.spines.bottom": True,
    "grid.color": "#a9d3f2",
    "grid.linewidth": "0.8",
    "text.color": "#333333",
    "axes.labelcolor": "#333333",
    "axes.labelweight": "black",      
    "xtick.color": "#333333",
    "ytick.color": "#333333",
    "font.size": 12,
    "axes.titleweight": "bold",       
    "legend.fontsize": 12,
    "legend.title_fontsize": 12,
}
plt.rcParams.update(light_style)
rcParams["figure.figsize"] = (18, 7)

import sys
import importlib
from pathlib import Path

SCRIPTS_PATH = Path.cwd().parents[2]

if str(SCRIPTS_PATH) not in sys.path:
    sys.path.append(str(SCRIPTS_PATH))

import utils
importlib.reload(utils)
from utils import EVALUAR_METRICAS

Importing plotly failed. Interactive plots will not work.
Importing plotly failed. Interactive plots will not work.


In [2]:
BASE_PATH = Path("..", "..", "..", "..")
FOLDER_DATA = BASE_PATH / "datasets" / "eda_archivos_cont_clima_indices"

## Carga de los datos y división del conjunto de datos

Cargamos los datos.

In [3]:
df = pd.read_csv(FOLDER_DATA / "dataset_cont_clima_indices_limpio.csv")

Filtramos las columnas que realmente necesitamos y como ciudad elegimos únicamente Madrid. Dado que todo el trabajo exploratorio ya se encuentra realizado en el *notebook* para el modelo Prophet, nos quedamos únicamente con las variables exógenas definidas anteriormente.

In [4]:
# Listado de columnas seleccionadas
columnas = [
    'Start', 'CO (mg.m-3)', 'city', 'temperature_2m', 'snowfall',
    'relative_humidity_2m', 'precipitation', 'rain',
    'surface_pressure', 'cloudcover', 'windspeed_10m', 
    'shortwave_radiation', 'boundary_layer_height', 'NDVI', 'NDBI', 'Año'
]

# Filtrar por Madrid y seleccionar las columnas
df1 = df[df['city'] == 'Madrid'][columnas].copy()

# Resetear los índices para que empiecen desde 0
df1.reset_index(drop=True, inplace=True)

# Eliminamos la columna 'city'
df1.drop(columns=['city'], inplace=True)

In [5]:
# ==============================================================================
# División cronológica del conjunto de datos
# ==============================================================================

train = df1[df1["Año"] <= 2020].copy()

validation = df1[
    (df1["Año"] >= 2021) &
    (df1["Año"] <= 2022)
].copy()

test = df1[df1["Año"] >= 2023].copy()

print(f"Entrenamiento: {train['Start'].min()} -> {train['Start'].max()}")
print(f"Validación:    {validation['Start'].min()} -> {validation['Start'].max()}")
print(f"Prueba:        {test['Start'].min()} -> {test['Start'].max()}")

print()
print(f"Nº muestras entrenamiento: {len(train):,}")
print(f"Nº muestras validación:    {len(validation):,}")
print(f"Nº muestras prueba:        {len(test):,}")

Entrenamiento: 2013-01-01 00:00:00 -> 2020-12-31 23:00:00
Validación:    2021-01-01 00:00:00 -> 2022-12-31 23:00:00
Prueba:        2023-01-01 00:00:00 -> 2024-12-31 23:00:00

Nº muestras entrenamiento: 70,128
Nº muestras validación:    17,520
Nº muestras prueba:        17,544


Para continuar con la forma en que Prophet denominaba a la variable objetivo y la columna temporal, renombramos las fechas por *ds* t la variable objetivo por *y*.

In [6]:
train = train.rename(columns={"Start": "ds", "CO (mg.m-3)": "y"})
validation = validation.rename(columns={"Start": "ds", "CO (mg.m-3)": "y"})
test = test.rename(columns={"Start": "ds", "CO (mg.m-3)": "y"})

In [7]:
variables_exogenas= [
    "temperature_2m",
    "relative_humidity_2m",
    "precipitation",
    "surface_pressure",
    "cloudcover",
    "windspeed_10m",
    "shortwave_radiation",
    "boundary_layer_height",
    "NDVI",
    "NDBI"
]

Nos quedamos exclusivamente con las variables necesarias para el modelado.

In [ ]:
# Lista de columnas que quieres mantener
columnas = ['ds', 'y'] + variables_exogenas

train = train[columnas]
validation = validation[columnas]
test = test[columnas]

# Eliminamos la primera fila del conjunto de entrenamiento
train = train.iloc[1:].reset_index(drop=True)

## Selección de los hiperparámetros

Para determinar los órdenes $(p,d,q)$ y $(P,D,Q)$ del modelo SARIMAX, realizamos una búsqueda automática de hiperparámetros mediante la función `auto_arima` del paquete `pmdarima`. Al trabajar con una serie temporal de frecuencia horaria, el período estacional $s$ se establece en función de la estacionalidad identificada previamente en la serie. Por ejemplo, para una estacionalidad diaria se utiliza $s=24$, mientras que para una estacionalidad semanal se emplea $s=168$. En caso de no existir una componente estacional relevante, el modelo se ajusta sin componente estacional.

La selección de los hiperparámetros se lleva a cabo en **dos etapas**. En primer lugar, aplicamos `auto_arima` exclusivamente sobre el conjunto de entrenamiento, proporcionando tanto la variable objetivo como las variables exógenas correspondientes a dicho período. La búsqueda se realiza de forma independiente utilizando tres criterios de información: **AIC**, **AICc** y **BIC**. De esta forma, obtenemos tres configuraciones candidatas, correspondientes a los modelos que minimizan cada uno de estos criterios.

Durante esta búsqueda, los órdenes de diferenciación regular $d$ y estacional $D$ son determinados automáticamente por `auto_arima`. Para los términos autorregresivos y de media móvil, tanto regulares como estacionales, se permite explorar valores comprendidos entre 0 y 8 para $p$, $q$, $P$ y $Q$. La búsqueda se realiza mediante el procedimiento `stepwise`, que explora de forma iterativa las configuraciones más prometedoras sin necesidad de evaluar exhaustivamente todas las combinaciones posibles.

En segundo lugar, cada una de las tres configuraciones candidatas obtenidas se vuelve a ajustar mediante la clase `SARIMAX` de `statsmodels`, utilizando exclusivamente el conjunto de entrenamiento. Para cada modelo se generan predicciones sobre el conjunto de validación, proporcionando también los valores observados de las variables exógenas correspondientes a dicho período.

A continuación, calculamos el error cuadrático medio (MSE) de las predicciones obtenidas por cada uno de los tres modelos candidatos. La configuración definitiva se selecciona como aquella que obtiene el menor MSE sobre el conjunto de validación. De este modo, los criterios AIC, AICc y BIC permiten realizar una primera selección basada únicamente en el conjunto de entrenamiento, mientras que el conjunto de validación permite determinar cuál de los candidatos presenta una mejor capacidad predictiva sobre observaciones no empleadas durante su ajuste.


In [9]:
def BUSQUEDA_CONFIGURACIONES_SARIMAX(
    train,
    validation,
    variables_exogenas,
    periodo_estacional=24
):
    """
    Busca automáticamente tres configuraciones SARIMAX:
    - La que minimiza el AIC.
    - La que minimiza el AICc.
    - La que minimiza el BIC.

    Las búsquedas se realizan exclusivamente sobre el conjunto
    de entrenamiento mediante auto_arima.

    Posteriormente, las tres configuraciones seleccionadas se
    ajustan sobre entrenamiento y se evalúan sobre validación.
    La configuración final será aquella que presente el menor
    MSE sobre el conjunto de validación.

    Parámetros
    ----------
    train : pd.DataFrame
        Conjunto de entrenamiento.

    validation : pd.DataFrame
        Conjunto de validación.

    variables_exogenas : list
        Variables exógenas empleadas por el modelo.

    periodo_estacional : int, default=24
        Periodicidad de la componente estacional.

    Retorna
    -------
    resultados : pd.DataFrame
        Resultados de los modelos seleccionados mediante
        AIC, AICc y BIC.

    mejor_configuracion : dict
        Configuración con menor MSE sobre validación.

    mejor_modelo :
        Modelo SARIMAX ajustado correspondiente a la
        mejor configuración.
    """

    # ==========================================================================
    # Variable objetivo
    # ==========================================================================

    y_train = train["y"]
    y_validation = validation["y"]

    # ==========================================================================
    # Variables exógenas
    # ==========================================================================

    X_train = train[variables_exogenas]
    X_validation = validation[variables_exogenas]

    # ==========================================================================
    # Criterios de información
    # ==========================================================================

    criterios = ["aic", "aicc", "bic"]

    configuraciones = []

    # ==========================================================================
    # Búsqueda automática de configuraciones
    # ==========================================================================

    for criterio in criterios:

        modelo_auto = auto_arima(
            y=y_train,
            X=X_train,

            # ==============================================================
            # Componente estacional
            # ==============================================================

            seasonal=True,
            m=periodo_estacional,

            # ==============================================================
            # Diferenciación
            # ==============================================================

            d=None,
            D=None,

            # ==============================================================
            # Búsqueda de órdenes
            # ==============================================================

            start_p=0,
            start_q=0,
            max_p=8,
            max_q=8,

            start_P=0,
            start_Q=0,
            max_P=8,
            max_Q=8,

            # ==============================================================
            # Criterio de selección
            # ==============================================================

            information_criterion=criterio,

            # ==============================================================
            # Configuración de la búsqueda
            # ==============================================================

            stepwise=True,
            suppress_warnings=True,
            error_action="ignore",
            trace=False
        )

        # ======================================================================
        # Órdenes seleccionados
        # ======================================================================

        order = modelo_auto.order
        seasonal_order = modelo_auto.seasonal_order

        configuraciones.append({
            "criterio": criterio.upper(),
            "order": order,
            "seasonal_order": seasonal_order
        })

    # ==========================================================================
    # Evaluación de las configuraciones sobre validación
    # ==========================================================================

    resultados = []

    mejor_mse = np.inf
    mejor_configuracion = None
    mejor_modelo = None

    for config in configuraciones:

        criterio = config["criterio"]
        order = config["order"]
        seasonal_order = config["seasonal_order"]

        try:

            # ==============================================================
            # Definición del modelo SARIMAX
            # ==============================================================

            modelo = SARIMAX(
                endog=y_train,
                exog=X_train,
                order=order,
                seasonal_order=seasonal_order,
                enforce_stationarity=False,
                enforce_invertibility=False
            )

            # ==============================================================
            # Ajuste exclusivamente sobre entrenamiento
            # ==============================================================

            modelo_ajustado = modelo.fit(
                disp=False
            )

            # ==============================================================
            # Predicción sobre validación
            # ==============================================================

            prediccion = modelo_ajustado.get_forecast(
                steps=len(validation),
                exog=X_validation
            )

            y_pred = np.asarray(
                prediccion.predicted_mean
            )

            # ==============================================================
            # MSE de validación
            # ==============================================================

            mse_validacion = metrics.mean_squared_error(
                y_validation,
                y_pred
            )

            # ==============================================================
            # AICc
            # ==============================================================

            n = modelo_ajustado.nobs
            k = len(modelo_ajustado.params)

            if n - k - 1 > 0:

                aicc = (
                    modelo_ajustado.aic
                    +
                    (2 * k * (k + 1)) / (n - k - 1)
                )

            else:

                aicc = np.nan

            # ==============================================================
            # Almacenar resultados
            # ==============================================================

            resultados.append({
                "criterio_seleccion": criterio,

                "p": order[0],
                "d": order[1],
                "q": order[2],

                "P": seasonal_order[0],
                "D": seasonal_order[1],
                "Q": seasonal_order[2],
                "s": seasonal_order[3],

                "AIC": modelo_ajustado.aic,
                "AICc": aicc,
                "BIC": modelo_ajustado.bic,

                "MSE_validacion": mse_validacion
            })

            # ==============================================================
            # Selección mediante MSE de validación
            # ==============================================================

            if mse_validacion < mejor_mse:

                mejor_mse = mse_validacion

                mejor_configuracion = {
                    "criterio_seleccion": criterio,

                    "p": order[0],
                    "d": order[1],
                    "q": order[2],

                    "P": seasonal_order[0],
                    "D": seasonal_order[1],
                    "Q": seasonal_order[2],
                    "s": seasonal_order[3]
                }

                mejor_modelo = modelo_ajustado

        except Exception:
            pass

    # ==========================================================================
    # DataFrame de resultados
    # ==========================================================================

    resultados = pd.DataFrame(resultados)

    resultados.sort_values(
        by="MSE_validacion",
        ascending=True,
        inplace=True
    )

    resultados.reset_index(
        drop=True,
        inplace=True
    )

    # ==========================================================================
    # Resultado final
    # ==========================================================================

    print("Mejores parámetros:")
    print()

    print(f"s = {periodo_estacional}")
    print(f"p = {mejor_configuracion['p']}")
    print(f"d = {mejor_configuracion['d']}")
    print(f"q = {mejor_configuracion['q']}")
    print(f"P = {mejor_configuracion['P']}")
    print(f"D = {mejor_configuracion['D']}")
    print(f"Q = {mejor_configuracion['Q']}")

    print()

    print(
        f"Criterio de Selección: "
        f"{mejor_configuracion['criterio_seleccion']}"
    )

    print(
        f"MSE de validación: {mejor_mse:.6f}"
    )

    return (
        resultados,
        mejor_configuracion,
        mejor_modelo
    )

In [17]:
resultados_sarimax, mejor_configuracion_sarimax, mejor_modelo_sarimax = (
    BUSQUEDA_CONFIGURACIONES_SARIMAX(
        train=train,
        validation=validation,
        variables_exogenas=variables_exogenas,
        periodo_estacional=24
    )
)


Mejores parámetros:

s = 24
p = 0
d = 3
q = 3
P = 1
D = 2
Q = 2

Criterio de Selección: AIC
MSE de validación: 0.051417


In [11]:
mejores_parametros = {
    "s": 24,
    "p": 0,
    "d": 3,
    "q": 3,
    "P": 1,
    "D": 2,
    "Q": 2
}

In [12]:
def ENTRENAR_EVALUAR_SARIMAX(
    train,
    validation,
    test,
    variables_exogenas,
    mejores_parametros
):
    """
    Entrena y evalúa un modelo SARIMAX utilizando los mejores
    hiperparámetros obtenidos previamente.

    El modelo se ajusta utilizando conjuntamente los conjuntos
    de entrenamiento y validación y se evalúa exclusivamente
    sobre el conjunto de prueba.

    Parámetros
    ----------
    train : pd.DataFrame
        Conjunto de entrenamiento.

    validation : pd.DataFrame
        Conjunto de validación.

    test : pd.DataFrame
        Conjunto de prueba.

    variables_exogenas : list
        Variables exógenas empleadas por el modelo.

    mejores_parametros : dict
        Diccionario con los mejores hiperparámetros:
        p, d, q, P, D, Q y s.

    Retorna
    -------
    modelo_ajustado :
        Modelo SARIMAX final ajustado.

    resultados : dict
        Diccionario con las métricas de evaluación.

    predicciones : np.ndarray
        Predicciones realizadas sobre el conjunto de prueba.
    """

    # ==========================================================================
    # Unión de entrenamiento y validación
    # ==========================================================================

    train_validation = pd.concat(
        [train, validation],
        ignore_index=True
    )

    # ==========================================================================
    # Conversión y ordenación temporal
    # ==========================================================================

    train_validation["ds"] = pd.to_datetime(
        train_validation["ds"]
    )

    test_sarimax = test.copy()

    test_sarimax["ds"] = pd.to_datetime(
        test_sarimax["ds"]
    )

    train_validation.sort_values(
        by="ds",
        inplace=True
    )

    test_sarimax.sort_values(
        by="ds",
        inplace=True
    )

    train_validation.reset_index(
        drop=True,
        inplace=True
    )

    test_sarimax.reset_index(
        drop=True,
        inplace=True
    )

    # ==========================================================================
    # Variable objetivo
    # ==========================================================================

    y_train_validation = train_validation["y"]
    y_test = test_sarimax["y"]

    # ==========================================================================
    # Variables exógenas
    # ==========================================================================

    X_train_validation = train_validation[
        variables_exogenas
    ]

    X_test = test_sarimax[
        variables_exogenas
    ]

    # ==========================================================================
    # Hiperparámetros
    # ==========================================================================

    p = mejores_parametros["p"]
    d = mejores_parametros["d"]
    q = mejores_parametros["q"]

    P = mejores_parametros["P"]
    D = mejores_parametros["D"]
    Q = mejores_parametros["Q"]

    s = mejores_parametros["s"]

    # ==========================================================================
    # Definición del modelo SARIMAX
    # ==========================================================================

    modelo = SARIMAX(
        endog=y_train_validation,
        exog=X_train_validation,
        order=(p, d, q),
        seasonal_order=(P, D, Q, s),
        enforce_stationarity=False,
        enforce_invertibility=False
    )

    # ==========================================================================
    # Entrenamiento del modelo
    # ==========================================================================

    modelo_ajustado = modelo.fit(
        disp=False
    )

    # ==========================================================================
    # Predicción sobre el conjunto de prueba
    # ==========================================================================

    prediccion = modelo_ajustado.get_forecast(
        steps=len(test_sarimax),
        exog=X_test
    )

    predicciones = np.asarray(
        prediccion.predicted_mean
    )

    # ==========================================================================
    # Número de parámetros del modelo
    # ==========================================================================

    num_parametros = len(
        modelo_ajustado.params
    )

    # ==========================================================================
    # Evaluación del modelo
    # ==========================================================================

    resultados = EVALUAR_METRICAS(
        y_real=y_test,
        y_predicho=predicciones,
        num_parametros=num_parametros
    )

    return (
        modelo_ajustado,
        resultados,
        predicciones
    )

In [16]:
modelo_sarimax, resultados_sarimax, predicciones_sarimax = (
    ENTRENAR_EVALUAR_SARIMAX(
        train=train,
        validation=validation,
        test=test,
        variables_exogenas=variables_exogenas,
        mejores_parametros=mejores_parametros
    )
)

Resultados de la evaluación del modelo
--------------------------------------
Error absoluto medio (MAE): 0.391402
Error cuadrático medio (MSE): 0.051417
Raíz del error cuadrático medio (RMSE): 0.226753
Error porcentual absoluto medio (MAPE): 46.77 %
Raíz del error cuadrático medio normalizada (NRMSE): 66.04 %
